# Lab 6 (lite): code generation in a real project

Lab 6 budget: $0.30, or nothing beyond a Claude subscription

**Every other lab calls a model from a cell. This one does not.**

Scenario 2, code generation with Claude Code. The repository is real, the client is the
`claude` CLI in a terminal, and this notebook builds the working copy and holds the prompts
you type.

Three routes, in one session: direct execution, plan mode, and a fork.

**Nothing in this notebook runs a model.** Run it after Chapter 19.

## 1. The project, and your working copy

`mycorp/` sits next to this notebook, checked in, so you can read it before a session touches
it: the refund path of the MyCorp online shop, seven modules and two decision records.

The cell below copies it and gives the copy a history, **two commits and a `lab-start` tag**,
so `git diff lab-start` shows everything you have done at any point. `REBUILD = True` throws
the copy away and starts again. That is the reset.

![Lab 6: code generation](../diagrams/lab-06-code-generation.png)

In [ ]:
import labkit

lab = labkit.start(model_env=None, credential="none")

import mycorp_lab

REBUILD = False          # True throws the working copy away, losing anything in it

REPO, git = mycorp_lab.build(lab, rebuild=REBUILD)

## 2. What the project tells Claude, and the one thing nobody decided

**`CLAUDE.md` is the first thing a session in this directory reads, and it is guidance rather
than enforced configuration.**

| Convention | What it decides later |
|---|---|
| A validator returns a message and never raises | Whether the first fix coerces the string or rejects it |
| The gateway is the only network boundary | Which functions can be tested without patching |

`.claude/settings.json` is the other half of what ships with the project, and that one is
enforced rather than advisory: a narrow allow list, so a session can edit, run the tests and
commit without a dialog in front of each one. It is checked in, so everyone gets the same
answer, and `git push` is denied in it.

Then the decision records. **ADR-0007 says in as many words that no decision has been taken**,
which is the whole reason section 4 goes to plan mode.

In [ ]:
mycorp_lab.project_docs(REPO)

## 3. Route by shape, before you type

**Write the route down before each task, because deciding afterwards is not deciding.** The
test is CALM: **c**omplex architecture, **a**lternative approaches, **l**arge file count,
**m**ulti-step exploration. Any one of them true means plan mode.

| Ticket | Shape | Route |
|---|---|---|
| TKT-0053, the string amount | One file, a trace that names the line | Direct execution |
| TKT-0054, tell the customer | Two defensible designs, no decision taken | Plan mode |

The cell below is the failure you are about to hand over. **A stack trace that names a file, a
function and a line is the clearest signal there is that a task wants direct execution**, because
there is nothing to discover and nothing to choose between. Hand it over as it is rather than
describing it.

In [ ]:
print(mycorp_lab.pytest_tail(REPO, "tests/test_refund_amount.py"))

## 4. Direct execution and plan mode

Open a terminal beside this notebook, and keep this one session for the rest of the lab.

```bash
cd workspace/mycorp
claude --session-id 6b1e0e4a-0000-4000-8000-000000000006
```

Pinning the id is worth the extra characters. Section 5 forks this session, and a fixed id
makes that one line to copy rather than a picker to read.

### One. Direct execution, because the trace names the line

> `tests/test_refund_amount.py` fails with `TypeError: '>' not supported between instances of
> 'str' and 'float'`, raised inside `RefundAmountValidator.check` in
> `shop/validators/refund_amount.py`. The portal-initiated refund path sends `refund_amount`
> as a JSON string, for example `"12.50"`. Read `shop/validators/__init__.py` first, then fix
> the validator so all three tests pass. Do not coerce the string to a number: the pattern in
> this project says a validator returns a message for a value it will not accept, and never
> raises. One file. No plan.

CALM has nothing true here, which is what makes it direct execution. The judgement the ticket
hides is that **coercing would also make the test pass**. `shop/validators/__init__.py` is what
says it is wrong.

```bash
git add -A && git commit -m "TKT-0053: return a message for a non-numeric refund amount"
```

### Two. Plan mode, because nobody has decided

**Plan mode has two justifications, and only one of them survives a small repository.** Blast
radius is the famous one, and it does not apply here: this project is seven modules and you
could read all of it. So say the other one out loud. You are not in plan mode because of size,
you are in plan mode because nobody has decided yet, and that does not get cheaper as the
repository gets smaller.

Shift+Tab cycles into plan mode. Read the indicator rather than counting presses, because the
stops it offers vary with your settings.

> Read `docs/adr/ADR-0007-refund-notifications.md`. It says plainly that no decision has been
> taken. Plan TKT-0054 both ways: the refunds service sending the notification itself on the
> way out of `send_refund`, and writing a row to `shop/outbox.py` for `notifications-worker`
> to drain. At most eight lines per design. For each one, give the failure mode when the
> notification path is down, and name the constraint in the ADR that decides it. Recommend one
> and say why. No edits.

**The reason it gives is the deliverable, not the design.** Nothing else happens in this
session, so leaving plan mode on costs you nothing.

In [ ]:
print(git("log", "--oneline"))
print()
print(git("diff", "--stat", "lab-start") or "nothing changed since lab-start yet")

## 5. Fork, the easy way

**A session is a transcript on disk, not a memory.** That is what makes a fork cheap: it
replays the same transcript under a new id, so you can take a second look without disturbing
the first.

The cell below prints the sessions recorded for this working copy, and the fork command ready
to copy.

```bash
claude --resume 6b1e0e4a-0000-4000-8000-000000000006 --fork-session
```

> Take the design you recommended for TKT-0054 and write it to
> `notes/fork-design.md` under three headings: the change, the failure mode, what a test
> would assert. No code.

**Re-run the cell below afterwards.** The new id is on disk, the original is untouched, and
you did not have to re-explain anything: it started from the same transcript.

In [ ]:
mycorp_lab.sessions(REPO)

## What you built

| The move | Where it happened |
|---|---|
| The repository, and its CLAUDE.md | Sections 1 and 2 |
| Direct execution, for a small stack-trace fix | Section 4, step one |
| Plan mode, for a decision nobody has taken | Section 4, step two |
| Fork, a second look from the same baseline | Section 5 |

**The decision to carry out of here is one sentence long.** Given a task and a codebase, say
whether to plan first or start typing, and say it before you type.